# Time Series with LSTM
## Week 6 — Day 3 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Use an LSTM model to analyze and predict household power consumption time-series data.

**Parts:**
1. Data Import & Initial Exploration
2. Handling Missing Values
3. Data Visualization
4. Data Preprocessing for LSTM
5. Building an LSTM Model
6. Training & Evaluating the LSTM Model

> **Run on Google Colab with GPU enabled** — Runtime → Change runtime type → GPU

## Setup — Install & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow  {tf.__version__} ✓")
print(f"NumPy       {np.__version__} ✓")
print(f"Pandas      {pd.__version__} ✓")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

tf.random.set_seed(42)
np.random.seed(42)

## Download Dataset

In [ ]:
import urllib.request
import zipfile
import os

DATASET_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
ZIP_PATH    = "household_power_consumption.zip"
CSV_PATH    = "household_power_consumption.txt"

if not os.path.exists(CSV_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('.')
    print("Dataset extracted ✓")
else:
    print("Dataset already present ✓")

print(f"File size: {os.path.getsize(CSV_PATH) / 1e6:.1f} MB")

## Part 1 — Data Import & Initial Exploration

In [ ]:
# Load dataset — separator is ';', missing values are encoded as '?'
df = pd.read_csv(
    CSV_PATH,
    sep=';',
    na_values=['?'],
    parse_dates={'Datetime': ['Date', 'Time']},
    dayfirst=True,
    infer_datetime_format=True,
    index_col='Datetime',
    low_memory=False
)

# Display first few rows
print("First 5 rows:")
display(df.head())

print(f"\nDataset shape: {df.shape}")
print(f"  Rows    : {df.shape[0]:,}  (1-minute readings)")
print(f"  Columns : {df.shape[1]}")
print(f"\nDate range: {df.index.min()}  →  {df.index.max()}")
print(f"  Duration: ~{(df.index.max() - df.index.min()).days / 365:.1f} years")

print("\nColumn data types:")
print(df.dtypes)

print("\nBasic statistics:")
display(df.describe())

## Part 2 — Handling Missing Values

In [ ]:
# Identify missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Missing values per column:")
display(missing_report[missing_report['Missing Count'] > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum():,}")

# Fill missing values with column mean
df.fillna(df.mean(numeric_only=True), inplace=True)

# Verify no more missing values
print("\nAfter filling with column means:")
remaining = df.isnull().sum().sum()
print(f"  Remaining missing values: {remaining} ✓" if remaining == 0 else f"  Still missing: {remaining}")
print("\nColumn means used for imputation:")
print(df.mean(numeric_only=True).round(4))

## Part 3 — Data Visualization

In [ ]:
# --- Plot 1: Global_active_power — daily sum and mean ---
gap_daily_sum  = df['Global_active_power'].resample('D').sum()
gap_daily_mean = df['Global_active_power'].resample('D').mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(gap_daily_sum.index, gap_daily_sum.values,
             color='#4e91d4', lw=0.8, alpha=0.9)
axes[0].fill_between(gap_daily_sum.index, gap_daily_sum.values,
                     alpha=0.2, color='#4e91d4')
axes[0].set_title('Daily Sum — Global Active Power (kW·min)', fontweight='bold')
axes[0].set_ylabel('Sum (kW·min)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(gap_daily_mean.index, gap_daily_mean.values,
             color='#e05c5c', lw=0.8, alpha=0.9)
axes[1].fill_between(gap_daily_mean.index, gap_daily_mean.values,
                     alpha=0.2, color='#e05c5c')
axes[1].set_title('Daily Mean — Global Active Power (kW)', fontweight='bold')
axes[1].set_ylabel('Mean (kW)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Global Active Power — Daily Resampled', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lstm_plot1_global_active_power.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 1 saved ✓")

In [ ]:
# --- Plot 2: Global_intensity — daily mean ± std ---
gi_daily_mean = df['Global_intensity'].resample('D').mean()
gi_daily_std  = df['Global_intensity'].resample('D').std()

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(gi_daily_mean.index, gi_daily_mean.values,
        color='#5cb85c', lw=1, label='Daily Mean')
ax.fill_between(gi_daily_mean.index,
                gi_daily_mean.values - gi_daily_std.values,
                gi_daily_mean.values + gi_daily_std.values,
                alpha=0.3, color='#5cb85c', label='± 1 Std Dev')

ax.set_title('Global Intensity — Daily Mean ± Standard Deviation', fontweight='bold', fontsize=12)
ax.set_xlabel('Date'); ax.set_ylabel('Global Intensity (A)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('lstm_plot2_global_intensity.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 2 saved ✓")

## Part 4 — Data Preprocessing for LSTM

In [ ]:
# Resample to hourly means to reduce sequence length (2M rows → ~47k)
df_hourly = df.resample('H').mean()
print(f"Resampled to hourly: {df_hourly.shape[0]:,} rows × {df_hourly.shape[1]} columns")

# Use Global_active_power as target for univariate prediction
series = df_hourly[['Global_active_power']].copy()

# --- Normalize with MinMaxScaler ---
scaler = MinMaxScaler(feature_range=(0, 1))
series_scaled = scaler.fit_transform(series)
print(f"Scaled range: [{series_scaled.min():.2f}, {series_scaled.max():.2f}]")

# --- Train / Test split (80% / 20%) ---
TRAIN_RATIO = 0.8
split_idx   = int(len(series_scaled) * TRAIN_RATIO)
train_data  = series_scaled[:split_idx]
test_data   = series_scaled[split_idx:]

print(f"\nTrain samples: {len(train_data):,}  ({TRAIN_RATIO*100:.0f}%)")
print(f"Test  samples: {len(test_data):,}  ({(1-TRAIN_RATIO)*100:.0f}%)")

# --- Create sequences for LSTM ---
LOOK_BACK = 24  # use past 24 hours to predict next hour

def create_sequences(data, look_back):
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, LOOK_BACK)
X_test,  y_test  = create_sequences(test_data,  LOOK_BACK)

# Reshape for LSTM: (samples, time_steps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"\nAfter sequence creation (look_back={LOOK_BACK}):")
print(f"  X_train : {X_train.shape}  (samples, time_steps, features)")
print(f"  y_train : {y_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_test  : {y_test.shape}")

## Part 5 — Building an LSTM Model

In [ ]:
model = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
], name='lstm_power_model')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

print("\nArchitecture rationale:")
print("  • LSTM(64, return_sequences=True) : first layer returns full sequence → feeds second LSTM")
print("  • Dropout(0.2) after each LSTM    : regularization against overfitting on time patterns")
print("  • LSTM(32, return_sequences=False): second layer returns only last hidden state")
print("  • Dense(16) → Dense(1)            : regression head — predicts next-hour power consumption")
print("  • Loss: MSE — standard for regression; Adam adapts lr per parameter")

## Part 6 — Training & Evaluating the LSTM Model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

EPOCHS     = 30
BATCH_SIZE = 64

print(f"Training LSTM for up to {EPOCHS} epochs (batch_size={BATCH_SIZE})...")
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)
print("Training complete ✓")

In [ ]:
# --- Training & Validation Loss plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs_ran, history.history['loss'],     label='Train Loss', color='#4e91d4', lw=2)
axes[0].plot(epochs_ran, history.history['val_loss'], label='Val Loss',   color='#e05c5c', lw=2, linestyle='--')
axes[0].set_title('MSE Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history.history['mae'],     label='Train MAE', color='#4e91d4', lw=2)
axes[1].plot(epochs_ran, history.history['val_mae'], label='Val MAE',   color='#e05c5c', lw=2, linestyle='--')
axes[1].set_title('Mean Absolute Error', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (scaled)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lstm_plot3_training_loss.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 3 saved ✓")

# --- Evaluation ---
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest MSE (scaled) : {test_loss:.6f}")
print(f"Test MAE (scaled) : {test_mae:.6f}")

# --- Inverse-scale predictions for interpretable metrics ---
y_pred_scaled = model.predict(X_test, verbose=0)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_true = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
print(f"\nTest RMSE (kW) : {rmse:.4f}")
print(f"Test MAE  (kW) : {mae:.4f}")

# --- Predictions vs Actual plot (last 7 days of test set) ---
PLOT_HOURS = 24 * 7
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_true[-PLOT_HOURS:],  label='Actual',    color='#4e91d4', lw=1.5)
ax.plot(y_pred[-PLOT_HOURS:],  label='Predicted', color='#e05c5c', lw=1.5, linestyle='--')
ax.set_title('LSTM Predictions vs Actual — Last 7 Days of Test Set', fontweight='bold')
ax.set_xlabel('Hours'); ax.set_ylabel('Global Active Power (kW)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lstm_plot4_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 4 saved ✓")

# --- Error distribution ---
errors = (y_true - y_pred).flatten()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(errors, bins=60, color='#4e91d4', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0].set_title('Prediction Error Distribution', fontweight='bold')
axes[0].set_xlabel('Error (kW)'); axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_true[-2000:], y_pred[-2000:], alpha=0.2, s=5, color='#4e91d4')
lim = max(y_true.max(), y_pred.max())
axes[1].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_title('Actual vs Predicted (scatter)', fontweight='bold')
axes[1].set_xlabel('Actual (kW)'); axes[1].set_ylabel('Predicted (kW)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Error Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('lstm_plot5_error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 5 saved ✓")

print(f"\n{'='*50}")
print(f"  LSTM MODEL — FINAL RESULTS")
print(f"{'='*50}")
print(f"  Architecture : LSTM(64) → Dropout → LSTM(32) → Dense(1)")
print(f"  Look-back    : {LOOK_BACK} hours")
print(f"  Test RMSE    : {rmse:.4f} kW")
print(f"  Test MAE     : {mae:.4f} kW")
print(f"{'='*50}")
print("\nKey takeaways:")
print("  1. LSTM captures temporal dependencies in power consumption patterns")
print("  2. Resampling to hourly reduces noise and training time")
print("  3. MinMaxScaler is essential — LSTM gradients are sensitive to scale")
print("  4. Two stacked LSTM layers learn both short-term and longer-term patterns")
print("  5. EarlyStopping prevents overfitting on the sequential training data")